# Frame averaging to a target frame rate

Averages every N frames of a lazy array to downsample from the source `fs` to a desired `target_hz`.

1. Read raw scanimage tiff
2. Average frames to achieve 4hz frame-rate
3. Save to .zarr (applies scanphase correction here)
4. Pass path to lsp.pipeline()
5. Repeat the same for the raw scanimage tiff without averaging frames

In [9]:
input_data_path = r"D:/Will_snyder"
output_data_path = r"D:/Will_snyder/frame_avg"

In [10]:
import numpy as np
import mbo_utilities as mbo
import lbm_suite2p_python as lsp

arr = mbo.imread("D:/Will_snyder")
fs = arr.metadata["fs"]
print(arr.shape, arr.dims, f"fs={fs} Hz")

(6979, 1, 14, 448, 448) ('T', 'C', 'Z', 'Y', 'X') fs=17.07 Hz


In [2]:
mbo.__version__, lsp.__version__

('3.0.1', '3.0.2')

You can perform operations on lazy-arrays as if they were in-memory numpy arrays.

In [3]:
def averaging_factor(fs: float, target_hz: float) -> int:
    n = int(round(fs / target_hz))
    return max(1, n)

target_hz = 4.0
N = averaging_factor(fs, target_hz)
effective_hz = fs / N
print(f"N={N}, effective_hz={effective_hz:.3f}")

N=4, effective_hz=4.268


In [4]:
def average_to_hz(
    lazy,
    target_hz: float,
    start: int = 0,
    stop: int | None = None,
    n_out: int | None = None,
    fs: float | None = None,
    dtype=np.float32,
    min_chunk_frames: int = 50,
):
    """
    Mean over N source frames from any mbo.LazyArray.
    """
    if fs is None:
        fs = lazy.metadata["fs"]
    N = averaging_factor(fs, target_hz)

    T = lazy.shape[0]
    if stop is None:
        stop = T
    max_blocks = (stop - start) // N
    if n_out is None:
        n_out = max_blocks
    n_out = min(n_out, max_blocks)

    blocks_per_chunk = max(1, -(-min_chunk_frames // N))  # ceil
    chunk_frames = blocks_per_chunk * N

    probe = np.asarray(lazy[start : start + N])
    out_shape = (n_out,) + probe.shape[1:]
    out = np.empty(out_shape, dtype=dtype)

    written = 0
    s = start
    while written < n_out:
        remaining = n_out - written
        n_blocks = min(blocks_per_chunk, remaining)
        e = s + n_blocks * N
        chunk = np.asarray(lazy[s:e])
        # split into n_blocks groups of N along axis 0 and average each
        reshaped = chunk.reshape((n_blocks, N) + chunk.shape[1:])
        out[written : written + n_blocks] = reshaped.mean(axis=1, dtype=dtype)
        written += n_blocks
        s = e

    return out, N, fs / N

In [5]:
averaged, N, eff_hz = average_to_hz(arr, target_hz=4.0)
print(f"averaged shape={averaged.shape}, N={N}, effective_hz={eff_hz:.3f}")

averaged shape=(1744, 1, 14, 448, 448), N=4, effective_hz=4.268


The averaged result is now in memory, we can pass that directly to `imwrite`.

When saving numpy arrays, pass `dim_order` if shape is anything other than `TCZYX`.

In [6]:
mbo.imwrite(
    averaged, #in memory
    "D:/Will_snyder/frame_avg",
    ext=".zarr",
    ome=True,
    dim_order="TCZYX",
    metadata={**dict(arr.metadata), "fs": eff_hz},
    overwrite=True,
)

Writing Zarr:   0%|          | 0/1744 [00:00<?, ?frames/s]

WindowsPath('D:/Will_snyder/frame_avg/tp00001-01744_ch01_zplane01-14_stack.zarr')

In [7]:
# you can open MBO Studio from a notebook like so
# add --metadata if you just want to preview parameters/metadata
!uv run mbo D:/Will_snyder/frame_avg/tp00001-01744_ch01_zplane01-14_stack.zarr


| Loading GUI...
                     


To silence this warning, use a fully namespaced name.
INFO - mbo.gui - Logger initialized.
INFO - mbo.gui - Data type: _SqueezeSingletonDims, is_mbo_scan: False
INFO - mbo.gui - Detected nz=14, nc=1, n_views=1 from dims=('T', 'Z', 'Y', 'X')
INFO - mbo.gui - Viewer: Time Series Viewer
INFO - mbo.gui - Starting zstats computation...
INFO - mbo.gui - Saved stats to array 1 metadata


In [8]:
lsp.pipeline(
    r"D:/Will_snyder/frame_avg/tp00001-01744_ch01_zplane01-14_stack.zarr",
    r"D:/Will_snyder/frame_avg/suite2p"
)

Loading input to determine dimensions...
Delegating to run_volume (volumetric input detected)...
Processing 14 planes in volume (Total planes: 14)
Output: D:\Will_snyder\frame_avg\suite2p

--- Volume Step: Plane 1 ---
Writing binary to D:\Will_snyder\frame_avg\suite2p\zplane01_tp00001-01744...


suite2p.pipeline_s2p: NOTE: applying default classifier: C:\Users\RBO\.suite2p\classifiers\classifier_user.npy
suite2p.pipeline_s2p: ----------- REGISTRATION
suite2p.registration.register: registering 1 channels
suite2p.registration.register: Reference frame, 2.38 sec.
suite2p.registration.register: Registering 1744 frames in 18 batches
suite2p.registration.register: 100%|##########| 18/18 [00:03<00:00,  4.57it/s]
suite2p.pipeline_s2p: ----------- Total 6.52 sec
suite2p.pipeline_s2p: Registration metrics, 8.19 sec.
suite2p.pipeline_s2p: ----------- ROI DETECTION
suite2p.pipeline_s2p: Excluding 0 bad frames from detection
suite2p.detection.detect: Binning movie in chunks of 04 frames
suite2p.detection.detect: 100%|##########| 4/4 [00:00<00:00,  6.01it/s]
suite2p.detection.detect: Binned movie of size [436,440,442] created in 0.67 sec.
suite2p.detection.sparsedetect: NOTE: estimated spatial scale ~6 pixels, time epochs 1.00, threshold 5.00 
suite2p.detection.sparsedetect: max_ROIs set to

  Computing dF/F...
  Computing ROI statistics...
Plotting results for 622 accepted / 1366 rejected ROIs

--- Volume Step: Plane 2 ---
Writing binary to D:\Will_snyder\frame_avg\suite2p\zplane02_tp00001-01744...


suite2p.pipeline_s2p: NOTE: applying default classifier: C:\Users\RBO\.suite2p\classifiers\classifier_user.npy
suite2p.pipeline_s2p: ----------- REGISTRATION
suite2p.registration.register: registering 1 channels
suite2p.registration.register: Reference frame, 2.12 sec.
suite2p.registration.register: Registering 1744 frames in 18 batches
suite2p.registration.register: 100%|##########| 18/18 [00:03<00:00,  4.76it/s]
suite2p.pipeline_s2p: ----------- Total 6.05 sec
suite2p.pipeline_s2p: Registration metrics, 8.05 sec.
suite2p.pipeline_s2p: ----------- ROI DETECTION
suite2p.pipeline_s2p: Excluding 0 bad frames from detection
suite2p.detection.detect: Binning movie in chunks of 04 frames
suite2p.detection.detect: 100%|##########| 4/4 [00:00<00:00,  5.89it/s]
suite2p.detection.detect: Binned movie of size [436,440,442] created in 0.68 sec.
suite2p.detection.sparsedetect: NOTE: estimated spatial scale ~6 pixels, time epochs 1.00, threshold 5.00 
suite2p.detection.sparsedetect: max_ROIs set to

  Computing dF/F...
  Computing ROI statistics...
Plotting results for 810 accepted / 1316 rejected ROIs

--- Volume Step: Plane 3 ---
Writing binary to D:\Will_snyder\frame_avg\suite2p\zplane03_tp00001-01744...


...

suite2p.pipeline_s2p: NOTE: applying default classifier: C:\Users\RBO\.suite2p\classifiers\classifier_user.npy
suite2p.pipeline_s2p: ----------- REGISTRATION
suite2p.registration.register: registering 1 channels
suite2p.registration.register: Reference frame, 2.22 sec.
suite2p.registration.register: Registering 1744 frames in 18 batches
suite2p.registration.register: 100%|##########| 18/18 [00:03<00:00,  4.70it/s]
suite2p.pipeline_s2p: ----------- Total 6.22 sec
suite2p.pipeline_s2p: Registration metrics, 7.66 sec.
suite2p.pipeline_s2p: ----------- ROI DETECTION
suite2p.pipeline_s2p: Excluding 0 bad frames from detection
suite2p.detection.detect: Binning movie in chunks of 04 frames
suite2p.detection.detect: 100%|##########| 4/4 [00:00<00:00,  5.96it/s]
suite2p.detection.detect: Binned movie of size [436,442,440] created in 0.67 sec.
suite2p.detection.sparsedetect: NOTE: estimated spatial scale ~6 pixels, time epochs 1.00, threshold 5.00 
suite2p.detection.sparsedetect: max_ROIs set to

  Computing dF/F...
  Computing ROI statistics...
Plotting results for 581 accepted / 2088 rejected ROIs

--- Volume Step: Plane 14 ---
Writing binary to D:\Will_snyder\frame_avg\suite2p\zplane14_tp00001-01744...


suite2p.pipeline_s2p: NOTE: applying default classifier: C:\Users\RBO\.suite2p\classifiers\classifier_user.npy
suite2p.pipeline_s2p: ----------- REGISTRATION
suite2p.registration.register: registering 1 channels
suite2p.registration.register: Reference frame, 2.13 sec.
suite2p.registration.register: Registering 1744 frames in 18 batches
suite2p.registration.register: 100%|##########| 18/18 [00:03<00:00,  4.65it/s]
suite2p.pipeline_s2p: ----------- Total 6.17 sec
suite2p.pipeline_s2p: Registration metrics, 8.02 sec.
suite2p.pipeline_s2p: ----------- ROI DETECTION
suite2p.pipeline_s2p: Excluding 0 bad frames from detection
suite2p.detection.detect: Binning movie in chunks of 04 frames
suite2p.detection.detect: 100%|##########| 4/4 [00:00<00:00,  6.02it/s]
suite2p.detection.detect: Binned movie of size [436,442,440] created in 0.66 sec.
suite2p.detection.sparsedetect: NOTE: estimated spatial scale ~6 pixels, time epochs 1.00, threshold 5.00 
suite2p.detection.sparsedetect: max_ROIs set to

  Computing dF/F...
  Computing ROI statistics...
Plotting results for 420 accepted / 1463 rejected ROIs

Genering volumetric statistics...


[WindowsPath('D:/Will_snyder/frame_avg/suite2p/zplane01_tp00001-01744/ops.npy'),
 WindowsPath('D:/Will_snyder/frame_avg/suite2p/zplane02_tp00001-01744/ops.npy'),
 WindowsPath('D:/Will_snyder/frame_avg/suite2p/zplane03_tp00001-01744/ops.npy'),
 WindowsPath('D:/Will_snyder/frame_avg/suite2p/zplane04_tp00001-01744/ops.npy'),
 WindowsPath('D:/Will_snyder/frame_avg/suite2p/zplane05_tp00001-01744/ops.npy'),
 WindowsPath('D:/Will_snyder/frame_avg/suite2p/zplane06_tp00001-01744/ops.npy'),
 WindowsPath('D:/Will_snyder/frame_avg/suite2p/zplane07_tp00001-01744/ops.npy'),
 WindowsPath('D:/Will_snyder/frame_avg/suite2p/zplane08_tp00001-01744/ops.npy'),
 WindowsPath('D:/Will_snyder/frame_avg/suite2p/zplane09_tp00001-01744/ops.npy'),
 WindowsPath('D:/Will_snyder/frame_avg/suite2p/zplane10_tp00001-01744/ops.npy'),
 WindowsPath('D:/Will_snyder/frame_avg/suite2p/zplane11_tp00001-01744/ops.npy'),
 WindowsPath('D:/Will_snyder/frame_avg/suite2p/zplane12_tp00001-01744/ops.npy'),
 WindowsPath('D:/Will_snyder

In [11]:
mbo.imwrite(
    arr,
    "D:/Will_snyder/frame_avg",
    output_suffix="default",
    ext=".zarr",
    overwrite=True,
)

Writing Zarr:   0%|          | 0/6979 [00:00<?, ?frames/s]

WindowsPath('D:/Will_snyder/frame_avg/tp00001-06979_ch01_zplane01-14_default.zarr')

In [13]:
!uv run mbo D:/Will_snyder/frame_avg/tp00001-06979_ch01_zplane01-14_default.zarr


| Loading GUI...
                     


To silence this warning, use a fully namespaced name.
INFO - mbo.gui - Logger initialized.
INFO - mbo.gui - Data type: _SqueezeSingletonDims, is_mbo_scan: False
INFO - mbo.gui - Detected nz=14, nc=1, n_views=1 from dims=('T', 'Z', 'Y', 'X')
INFO - mbo.gui - Viewer: Time Series Viewer
INFO - mbo.gui - Starting zstats computation...
INFO - mbo.gui - Saved stats to array 1 metadata
INFO - mbo.gui - Shortcut: 'm' (Metadata Viewer)


In [9]:
data_no_avg = mbo.imread(r"D:/demo/averaged_zarr/tp00001-01574_ch01_zplane01-14_default.zarr/")
data_no_avg.shape

(1574, 1, 14, 550, 448)

In [14]:
lsp.pipeline(
    r"D:/Will_snyder/frame_avg/tp00001-06979_ch01_zplane01-14_default.zarr",
    r"D:/Will_snyder/frame_avg/s2p_sparsery_defaults_noavg"
)

Loading input to determine dimensions...
Delegating to run_volume (volumetric input detected)...
Processing 14 planes in volume (Total planes: 14)
Output: D:\Will_snyder\frame_avg\s2p_sparsery_defaults_noavg

--- Volume Step: Plane 1 ---
Writing binary to D:\Will_snyder\frame_avg\s2p_sparsery_defaults_noavg\zplane01_tp00001-06979...


suite2p.pipeline_s2p: NOTE: applying default classifier: C:\Users\RBO\.suite2p\classifiers\classifier_user.npy
suite2p.pipeline_s2p: ----------- REGISTRATION
suite2p.registration.register: registering 1 channels
suite2p.registration.register: Reference frame, 2.30 sec.
suite2p.registration.register: Registering 6979 frames in 70 batches
suite2p.registration.register: 100%|##########| 70/70 [00:15<00:00,  4.54it/s]
suite2p.pipeline_s2p: ----------- Total 17.93 sec
suite2p.pipeline_s2p: Registration metrics, 12.79 sec.
suite2p.pipeline_s2p: ----------- ROI DETECTION
suite2p.pipeline_s2p: Excluding 0 bad frames from detection
suite2p.detection.detect: Binning movie in chunks of 17 frames
suite2p.detection.detect: 100%|##########| 14/14 [00:01<00:00,  7.26it/s]
suite2p.detection.detect: Binned movie of size [405,432,440] created in 1.93 sec.
suite2p.detection.sparsedetect: NOTE: estimated spatial scale ~6 pixels, time epochs 1.00, threshold 5.00 
suite2p.detection.sparsedetect: max_ROIs se

  Computing dF/F...
  Computing ROI statistics...
Plotting results for 601 accepted / 1193 rejected ROIs

--- Volume Step: Plane 2 ---
Writing binary to D:\Will_snyder\frame_avg\s2p_sparsery_defaults_noavg\zplane02_tp00001-06979...


suite2p.pipeline_s2p: NOTE: applying default classifier: C:\Users\RBO\.suite2p\classifiers\classifier_user.npy
suite2p.pipeline_s2p: ----------- REGISTRATION
suite2p.registration.register: registering 1 channels
suite2p.registration.register: Reference frame, 2.11 sec.
suite2p.registration.register: Registering 6979 frames in 70 batches
suite2p.registration.register: 100%|##########| 70/70 [00:15<00:00,  4.64it/s]
suite2p.pipeline_s2p: ----------- Total 17.37 sec
suite2p.pipeline_s2p: Registration metrics, 12.72 sec.
suite2p.pipeline_s2p: ----------- ROI DETECTION
suite2p.pipeline_s2p: Excluding 0 bad frames from detection
suite2p.detection.detect: Binning movie in chunks of 17 frames
suite2p.detection.detect: 100%|##########| 14/14 [00:01<00:00,  7.27it/s]
suite2p.detection.detect: Binned movie of size [405,436,440] created in 1.93 sec.
suite2p.detection.sparsedetect: NOTE: estimated spatial scale ~6 pixels, time epochs 1.00, threshold 5.00 
suite2p.detection.sparsedetect: max_ROIs se

  Computing dF/F...
  Computing ROI statistics...
Plotting results for 785 accepted / 1290 rejected ROIs

--- Volume Step: Plane 3 ---
Writing binary to D:\Will_snyder\frame_avg\s2p_sparsery_defaults_noavg\zplane03_tp00001-06979...


...

suite2p.pipeline_s2p: NOTE: applying default classifier: C:\Users\RBO\.suite2p\classifiers\classifier_user.npy
suite2p.pipeline_s2p: ----------- REGISTRATION
suite2p.registration.register: registering 1 channels
suite2p.registration.register: Reference frame, 2.29 sec.
suite2p.registration.register: Registering 6979 frames in 70 batches
suite2p.registration.register: 100%|##########| 70/70 [00:16<00:00,  4.18it/s]
suite2p.pipeline_s2p: ----------- Total 19.24 sec
suite2p.pipeline_s2p: Registration metrics, 15.56 sec.
suite2p.pipeline_s2p: ----------- ROI DETECTION
suite2p.pipeline_s2p: Excluding 0 bad frames from detection
suite2p.detection.detect: Binning movie in chunks of 17 frames
suite2p.detection.detect: 100%|##########| 14/14 [00:02<00:00,  5.19it/s]
suite2p.detection.detect: Binned movie of size [405,438,440] created in 2.70 sec.
suite2p.detection.sparsedetect: NOTE: estimated spatial scale ~6 pixels, time epochs 1.00, threshold 5.00 
suite2p.detection.sparsedetect: max_ROIs se

  Computing dF/F...
  Computing ROI statistics...
Plotting results for 627 accepted / 1623 rejected ROIs

--- Volume Step: Plane 14 ---
Writing binary to D:\Will_snyder\frame_avg\s2p_sparsery_defaults_noavg\zplane14_tp00001-06979...


suite2p.pipeline_s2p: NOTE: applying default classifier: C:\Users\RBO\.suite2p\classifiers\classifier_user.npy
suite2p.pipeline_s2p: ----------- REGISTRATION
suite2p.registration.register: registering 1 channels
suite2p.registration.register: Reference frame, 2.12 sec.
suite2p.registration.register: Registering 6979 frames in 70 batches
suite2p.registration.register: 100%|##########| 70/70 [00:16<00:00,  4.29it/s]
suite2p.pipeline_s2p: ----------- Total 18.64 sec
suite2p.pipeline_s2p: Registration metrics, 13.58 sec.
suite2p.pipeline_s2p: ----------- ROI DETECTION
suite2p.pipeline_s2p: Excluding 0 bad frames from detection
suite2p.detection.detect: Binning movie in chunks of 17 frames
suite2p.detection.detect: 100%|##########| 14/14 [00:02<00:00,  6.31it/s]
suite2p.detection.detect: Binned movie of size [405,436,438] created in 2.23 sec.
suite2p.detection.sparsedetect: NOTE: estimated spatial scale ~6 pixels, time epochs 1.00, threshold 5.00 
suite2p.detection.sparsedetect: max_ROIs se

  Computing dF/F...
  Computing ROI statistics...
Plotting results for 423 accepted / 1186 rejected ROIs

Genering volumetric statistics...


[WindowsPath('D:/Will_snyder/frame_avg/s2p_sparsery_defaults_noavg/zplane01_tp00001-06979/ops.npy'),
 WindowsPath('D:/Will_snyder/frame_avg/s2p_sparsery_defaults_noavg/zplane02_tp00001-06979/ops.npy'),
 WindowsPath('D:/Will_snyder/frame_avg/s2p_sparsery_defaults_noavg/zplane03_tp00001-06979/ops.npy'),
 WindowsPath('D:/Will_snyder/frame_avg/s2p_sparsery_defaults_noavg/zplane04_tp00001-06979/ops.npy'),
 WindowsPath('D:/Will_snyder/frame_avg/s2p_sparsery_defaults_noavg/zplane05_tp00001-06979/ops.npy'),
 WindowsPath('D:/Will_snyder/frame_avg/s2p_sparsery_defaults_noavg/zplane06_tp00001-06979/ops.npy'),
 WindowsPath('D:/Will_snyder/frame_avg/s2p_sparsery_defaults_noavg/zplane07_tp00001-06979/ops.npy'),
 WindowsPath('D:/Will_snyder/frame_avg/s2p_sparsery_defaults_noavg/zplane08_tp00001-06979/ops.npy'),
 WindowsPath('D:/Will_snyder/frame_avg/s2p_sparsery_defaults_noavg/zplane09_tp00001-06979/ops.npy'),
 WindowsPath('D:/Will_snyder/frame_avg/s2p_sparsery_defaults_noavg/zplane10_tp00001-06979/o